# Bangladesh: 0123_xee_biomass_xgboost

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0123_xee_biomass_xgboost
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'forest' / '0123_xee_biomass_xgboost'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""GEE_0035_biomass_modelling.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-128: Biomass and Carbon Stack Modelling using XGBoost Regression, in Python API (Xee)

This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor



"""
import ee
import geemap
import xarray as xr
import xee
None
None
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
biomass = ee.ImageCollection('NASA/ORNL/biomass_carbon_density/v1').select('agb').first().rename('biomass')
ndvi = ee.ImageCollection('MODIS/061/MOD13A2').select('NDVI', 'EVI').filterDate('2010', '2011').mean().rename('ndvi', 'evi')
lai = ee.ImageCollection('MODIS/061/MOD15A2H').select('Lai_500m', 'Fpar_500m').filterDate('2010', '2011').mean().rename('lai', 'fpar')
temp = ee.ImageCollection('MODIS/061/MYD11A2').select('LST_Day_1km').filterDate('2010', '2011').mean().rename('temp')
pr = ee.ImageCollection('NASA/GPM_L3/IMERG_MONTHLY_V07').select('precipitation').filterDate('2010', '2011').sum().rename('pr')
lc = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1').filterDate('2010', '2011').first().rename('lc')
stack = ee.Image.cat([ndvi, lai, temp, pr, lc, biomass])
stack
ds = xr.open_dataset(stack, engine='ee', crs='EPSG:4326', scale=0.1, geometry=BD_ROI)
ds
df = ds.to_dataframe().dropna()
df
x = df[['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']]
y = df['biomass']
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
from xgboost import XGBRegressor
model = XGBRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'RMSE:{rmse}')
r2 = r2_score(y_test, y_pred)
print(f'R2:{r2}')
df['predicted_biomass'] = model.predict(x)
df
xarr = df.to_xarray().sortby('lon').sortby('lat')
xarr
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
plt.tight_layout()
xarr.biomass.plot(ax=ax[0], x='lon', y='lat', robust=True)
xarr.predicted_biomass.plot(ax=ax[1], x='lon', y='lat', robust=True)
time_start = ee.Date('2015')
time_end = ee.Date('2016')
ndvi = ee.ImageCollection('MODIS/061/MOD13A2').select('NDVI', 'EVI').filterDate(time_start, time_end).mean().rename('ndvi', 'evi')
lai = ee.ImageCollection('MODIS/061/MOD15A2H').select('Lai_500m', 'Fpar_500m').filterDate(time_start, time_end).mean().rename('lai', 'fpar')
temp = ee.ImageCollection('MODIS/061/MYD11A2').select('LST_Day_1km').filterDate(time_start, time_end).mean().rename('temp')
pr = ee.ImageCollection('NASA/GPM_L3/IMERG_MONTHLY_V07').select('precipitation').filterDate(time_start, time_end).sum().rename('pr')
lc = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1').filterDate(time_start, time_end).first().rename('lc')
stack2015 = ee.Image.cat([ndvi, lai, temp, pr, lc])
stack2015
ds2015 = xr.open_dataset(stack2015, engine='ee', crs='EPSG:4326', scale=0.1, geometry=BD_ROI)
df2015 = ds2015.to_dataframe().dropna()
df2015
df2015['biomass2015'] = model.predict(df2015[['ndvi', 'lai', 'temp', 'pr', 'lc', 'fpar', 'evi']])
df2015
xarr2015 = df2015.to_xarray().sortby('lon').sortby('lat')
xarr2015
xarr2015.biomass2015.plot(x='lon', y='lat', robust=True)


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'forest', '0123_xee_biomass_xgboost', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'forest')  # explicit upload; requires Drive OAuth scope
